# Prompt Preview

Ce notebook lit `input/swissdox_2025_tagged.csv`, **reproduit l'explosion par mot-clé de la pipeline 3** (une ligne par couple article/keyword, comme sur le cluster), puis produit un CSV avec :
- **`keyword`** : le mot-clé isolé pour cette ligne
- **`prompt_ready`** : le prompt complet à coller dans un chat LLM
- **`llm_answer`** : colonne vide à remplir manuellement avec la réponse du LLM (YES / NO)

La colonne `text` brute est exclue pour alléger le fichier.

In [ ]:
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve() / 'src'))
from run3_prompts import build_user_prompt, SYSTEM_PROMPT
from run3_config import build_mask

In [2]:
# Charger les données
df = pd.read_csv('../input/swissdox_2025_tagged.csv')
print(f"Articles chargés : {len(df):,}")
print(f"Colonnes : {df.columns.tolist()}")
df.head(2)

Articles chargés : 23,665
Colonnes : ['article_id', 'title', 'lead', 'text', 'pubtime', 'medium_code', 'medium_name', 'rubric', 'regional', 'doctype', 'doctype_description', 'language', 'char_count', 'dateline', 'matched_keywords']


,article_id,title,lead,text,pubtime,medium_code,medium_name,rubric,regional,doctype,doctype_description,language,char_count,dateline,matched_keywords
0,0001de76-bb46-1fd0-ff60-980d765e15a8,Mega-Angriff legt Schweizer Seiten lahm: Hacke...,NoName057(16) bekennt sich zu DDoS-Angriffen a...,NoName057(16) bekennt sich zu DDoS-Angriffen a...,2025-01-21,ZWAO,20 minuten online,NaN,NaN,WWE,Online medium,de,2666.0,NaN,Behörden
1,00027f05-690b-f341-5519-fcc50dbc5549,Martin Pfister envisage l’abandon du projet à ...,Le Conseiller fédéral envisage d’abandonner l’...,Le Conseiller fédéral envisage d’abandonner l’...,2025-07-05,ZWSO,20 minutes online,NaN,NaN,WWE,Online medium,fr,2813.0,NaN,Martin Pfister|Armée suisse


In [ ]:
# Reproduire l'explosion par mot-clé de la pipeline 3
# (une ligne par (article, keyword), comme run3_pipeline.py fait sur le cluster)
mask = build_mask(df, text_col='text')
rows = df[mask].copy()
rows['keyword'] = rows['matched_keywords'].str.split('|')
df_exploded = rows.explode('keyword').copy()
df_exploded['keyword'] = df_exploded['keyword'].str.strip()
df_exploded = df_exploded[df_exploded['keyword'] != ''].reset_index(drop=True)

print(f"Articles avec texte + mots-clés : {int(mask.sum()):,}")
print(f"Lignes après explosion (article, keyword) : {len(df_exploded):,}")
df_exploded[['article_id', 'matched_keywords', 'keyword']].head(5)

In [ ]:
# Construire la colonne prompt_ready — un prompt par (article, keyword)
df_exploded['prompt_ready'] = df_exploded.apply(
    lambda row: ((SYSTEM_PROMPT + "\n\n") if SYSTEM_PROMPT else "") + build_user_prompt(row, 'text'),
    axis=1
)

df_exploded['llm_answer'] = ''

print(f"Exemple de prompt_ready pour la première ligne :")
print("-" * 60)
print(df_exploded['prompt_ready'].iloc[0])

In [ ]:
# Supprimer la colonne text brute et exporter
cols_to_keep = [c for c in df_exploded.columns if c != 'text']
df_out = df_exploded[cols_to_keep]

out_path = Path('../data/raw/swissdox_prompts_run3.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_path, index=False)

print(f"Fichier exporté : {out_path}")
print(f"Colonnes : {df_out.columns.tolist()}")
print(f"Lignes : {len(df_out):,}")

In [ ]:
# Aperçu du résultat final
df_out[['article_id', 'medium_name', 'pubtime', 'matched_keywords', 'keyword', 'prompt_ready', 'llm_answer']].head(5)